# EDA: режимы электрической сети (35 узлов, 80 линий)

**Файлы:** `train_nodes_34.csv` (узлы), `train_lines_34.csv` (линии). Связь между ними: `ID_опыта`.

**План:**
0. Загрузка и общий обзор
1. Проверка гипотезы: если `regime_valid=0` хотя бы в одном узле опыта, то во всех узлах опыта = 0
2. Связь `Q_н` и `P_н` (`Q = k·P`?)
3. `P_г` в узлах 29 и 33 (и вообще по всем узлам, баланс, балансирующий узел)
4. Разделение на valid=1 / valid=0, сравнение распределений
5. Линии: что постоянно, что меняется, связь `I_max`, `I_доп`, `I_zarg`
6. Корреляции: входы (нагрузка/генерация/параметры линий) → выходы (V, Delta, I_max, I_zarg)
7. Что влияет на сходимость (классификация)
8. Итоги

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

NODES_PATH = "train_nodes_34.csv"
LINES_PATH = "train_lines_34.csv"

## 0. Загрузка и общий обзор

In [ ]:
def load(path):
    df = pd.read_csv(path)
    if df.shape[1] == 1:               # на случай разделителя ';'
        df = pd.read_csv(path, sep=";")
    df.columns = df.columns.str.strip()
    return df

nodes_raw, lines_raw = load(NODES_PATH), load(LINES_PATH)
print(nodes_raw.columns.tolist())
print(lines_raw.columns.tolist())

# ASCII-имена, чтобы удобнее писать код (порядок колонок как в ТЗ / на скриншотах)
nodes = nodes_raw.copy()
lines = lines_raw.copy()
nodes.columns = ["id", "node", "U_nom", "P_load", "Q_load", "P_gen", "V", "Delta", "valid"]
lines.columns = ["id", "line", "n_s", "n_e", "type", "R", "X", "B", "Kt", "I_max", "I_dop", "I_zarg"]

# относительное напряжение (о.е.) - удобно сравнивать узлы 110/220/500 кВ
nodes["V_pu"] = nodes["V"] / nodes["U_nom"]

In [ ]:
for name, df in [("nodes", nodes), ("lines", lines)]:
    print("=" * 30, name, df.shape)
    display(df.head(8))
    display(df.describe().T)
    na = df.isna().sum()
    print("NaN по колонкам:\n", na[na > 0] if (na > 0).any() else "пропусков нет")

In [ ]:
print("Опытов в nodes:", nodes.id.nunique(), "| в lines:", lines.id.nunique())
print("Множества ID совпадают:", set(nodes.id) == set(lines.id))
print("\nУзлов на опыт:");  print(nodes.groupby("id").size().value_counts())
print("\nЛиний на опыт:");  print(lines.groupby("id").size().value_counts())
print("\nДубликаты (id,node):", nodes.duplicated(["id", "node"]).sum(),
      "| (id,line):", lines.duplicated(["id", "line"]).sum())
print("\nУзлы:", sorted(nodes.node.unique()))
print("Номинальные напряжения:", nodes.groupby("U_nom").node.nunique().to_dict(), "(узлов на класс)")
REF = 999  # эталонный (балансирующий) узел: есть в линиях, но нет в таблице узлов
print("Концы линий вне таблицы узлов:", sorted((set(lines.n_s) | set(lines.n_e)) - set(nodes.node)), "-> эталонный узел", REF)
print("Линий, подключённых к 999:", lines[(lines.n_s == REF) | (lines.n_e == REF)].line.nunique())

## 1. Гипотеза: флаг `regime_valid` одинаков во всех узлах опыта

Если в опыте хоть в одном узле флаг = 0, то и во всех остальных = 0.

In [ ]:
g = nodes.groupby("id")["valid"]
exp = pd.DataFrame({"vmin": g.min(), "vmax": g.max(), "share_valid": g.mean(), "n_nodes": g.size()})
mixed = exp[exp.vmin != exp.vmax]

print(f"Опытов всего: {len(exp)}")
print(f"Опытов со СМЕШАННЫМ флагом (и 0 и 1 внутри опыта): {len(mixed)}")
if len(mixed):
    print("Гипотеза НЕ подтверждена, примеры:")
    display(mixed.head(10))
    display(mixed.share_valid.describe())
else:
    print("Гипотеза ПОДТВЕРЖДЕНА: флаг един для всего опыта.")

# флаг уровня опыта (если смешанный - считаем опыт невалидным, если есть хоть один 0)
exp_valid = exp["vmin"].rename("valid_exp")
print("\nДоля валидных опытов:")
print(exp_valid.value_counts(normalize=True).rename({0: "invalid (0)", 1: "valid (1)"}))
nodes = nodes.merge(exp_valid, left_on="id", right_index=True)
lines = lines.merge(exp_valid, left_on="id", right_index=True)

## 2. `Q_н` vs `P_н`: есть ли постоянный коэффициент (tgφ)?

In [ ]:
print("P_load<0:", (nodes.P_load < 0).sum(), "| Q_load<0:", (nodes.Q_load < 0).sum())
print("P!=0 & Q==0:", ((nodes.P_load != 0) & (nodes.Q_load == 0)).sum())
print("P==0 & Q!=0:", ((nodes.P_load == 0) & (nodes.Q_load != 0)).sum())
print("Узлов-строк с ненулевой нагрузкой:", (nodes.P_load != 0).mean().round(3))

def tg_stats(d):
    m = d.P_load != 0
    t = d.Q_load[m] / d.P_load[m]
    return pd.Series({
        "rows": len(d),
        "P_nonzero_share": m.mean(),
        "Q_nonzero_share": (d.Q_load != 0).mean(),
        "tg_mean": t.mean(), "tg_std": t.std(), "tg_min": t.min(), "tg_max": t.max(),
        "tg_unique(round4)": t.round(4).nunique() if len(t) else 0,
        "corr(P,Q)": d[["P_load", "Q_load"]].corr().iloc[0, 1] if m.any() else np.nan,
    })

per_node_tg = nodes.groupby("node").apply(tg_stats)
display(per_node_tg)

In [ ]:
nz = nodes[nodes.P_load != 0].copy()
nz["tg"] = nz.Q_load / nz.P_load

print("Глобально tg = Q/P:")
display(nz.tg.describe())
print("\nПо классам напряжения:")
display(nz.groupby("U_nom").tg.describe())

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
nz.tg.clip(-2, 2).hist(bins=100, ax=ax[0]); ax[0].set_title("Распределение tg=Q/P (clip ±2)")
sns.scatterplot(data=nz.sample(min(20000, len(nz)), random_state=0), x="P_load", y="Q_load",
                hue="U_nom", s=8, alpha=.5, ax=ax[1], palette="tab10"); ax[1].set_title("Q_н vs P_н")
per_node_tg["tg_std"].plot.bar(ax=ax[2]); ax[2].set_title("std(tg) по узлам (0 => Q=k·P строго)")
plt.tight_layout(); plt.show()

# Если k постоянный на узел - в этой таблице почти всё 1 уникальное значение
print("Узлов, где tg фактически константа (std<1e-3):",
      int((per_node_tg["tg_std"] < 1e-3).sum()), "из", int(per_node_tg["P_nonzero_share"].gt(0).sum()))

## 3. `P_г`: узлы 29 / 33 и общая картина генерации

In [ ]:
gen = nodes.groupby("node").agg(
    U_nom=("U_nom", "first"),
    P_gen_nonzero_share=("P_gen", lambda s: (s != 0).mean()),
    P_gen_min=("P_gen", "min"), P_gen_mean=("P_gen", "mean"), P_gen_max=("P_gen", "max"),
    P_load_nonzero_share=("P_load", lambda s: (s != 0).mean()),
    P_load_mean=("P_load", "mean"), P_load_max=("P_load", "max"),
    V_pu_std=("V_pu", "std"),
)
print("Узлы, где P_gen вообще бывает != 0:")
display(gen[gen.P_gen_nonzero_share > 0])
print("Узлы 29 и 33:")
display(gen.loc[[n for n in (29, 33) if n in gen.index]])

In [ ]:
sel = [n for n in (29, 33) if n in nodes.node.unique()]
fig, ax = plt.subplots(2, len(sel), figsize=(7 * len(sel), 8), squeeze=False)
for j, n in enumerate(sel):
    d = nodes[nodes.node == n]
    sns.histplot(data=d, x="P_gen", hue="valid_exp", bins=60, ax=ax[0, j], stat="density", common_norm=False)
    ax[0, j].set_title(f"P_г в узле {n} (по флагу)")
    sns.histplot(data=d, x="P_load", hue="valid_exp", bins=60, ax=ax[1, j], stat="density", common_norm=False)
    ax[1, j].set_title(f"P_н в узле {n} (по флагу)")
plt.tight_layout(); plt.show()

# P_г в 29 и 33 одновременно - есть ли связь между узлами
if len(sel) == 2:
    pg = nodes[nodes.node.isin(sel)].pivot(index="id", columns="node", values="P_gen")
    print("corr(P_gen_29, P_gen_33):", pg.corr().iloc[0, 1].round(4))
    print("P_gen!=0 одновременно в 29 и 33:", ((pg[29] != 0) & (pg[33] != 0)).mean().round(4))
    display(pg.describe())

In [ ]:
# Баланс мощности по опыту и поиск балансирующего узла
bal = nodes.groupby("id").agg(P_load=("P_load", "sum"), P_gen=("P_gen", "sum"),
                              valid=("valid_exp", "first"))
bal["P_gen_minus_load"] = bal.P_gen - bal.P_load
display(bal.groupby("valid")[["P_load", "P_gen", "P_gen_minus_load"]].describe().T)

# узлы с (почти) фиксированным V_pu среди валидных - кандидаты на PV/балансирующий узел
v_ok = nodes[nodes.valid_exp == 1].groupby("node").V_pu.agg(["mean", "std", "min", "max"])
print("Узлы с наименьшим разбросом V_pu (валидные режимы):")
display(v_ok.sort_values("std").head(8))

## 4. Разделение на valid = 1 / valid = 0

In [ ]:
v1 = nodes[nodes.valid_exp == 1]; v0 = nodes[nodes.valid_exp == 0]
print("Строк узлов valid=1:", len(v1), "| valid=0:", len(v0))

cols = ["P_load", "Q_load", "P_gen", "V", "V_pu", "Delta"]
display(pd.concat({"valid=1": v1[cols].describe().T, "valid=0": v0[cols].describe().T}, axis=1))

# что лежит в V/Delta у невалидных опытов: нули, NaN или мусор?
for c in ["V", "Delta"]:
    print(f"{c}: valid=0 -> NaN {v0[c].isna().mean():.3f}, нули {(v0[c]==0).mean():.3f}; "
          f"valid=1 -> NaN {v1[c].isna().mean():.3f}, нули {(v1[c]==0).mean():.3f}")

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(18, 8))
for a, c in zip(ax.ravel(), cols):
    d = nodes[[c, "valid_exp"]].replace([np.inf, -np.inf], np.nan).dropna()
    lo, hi = d[c].quantile([.005, .995])
    sns.histplot(data=d[d[c].between(lo, hi)], x=c, hue="valid_exp", bins=70, stat="density",
                 common_norm=False, ax=a)
    a.set_title(c)
plt.tight_layout(); plt.show()

## 5. Линии

Что **константа** по опытам (топология/параметры), а что **меняется** (результат расчёта)?

In [ ]:
static_cols = ["n_s", "n_e", "type", "R", "X", "B", "Kt", "I_dop"]
nun = lines.groupby("line")[static_cols].nunique()
print("Число линий:", lines.line.nunique())
print("Колонки, у которых значение меняется между опытами хотя бы у одной линии:")
print((nun > 1).sum()[lambda s: s > 0] if (nun > 1).any().any() else "все параметры строго постоянны")

line_info = lines.groupby("line")[["n_s", "n_e", "type", "R", "X", "B", "Kt", "I_dop"]].first()
display(line_info.head(15))
print("\nТипы ветвей:"); print(lines.groupby("type").agg(n_lines=("line", "nunique"),
      R=("R", "mean"), X=("X", "mean"), B=("B", "mean"), Kt=("Kt", "mean")))

In [ ]:
# Соотношение I_zarg = I_max / I_dop * 100 ?
chk = lines[lines.I_dop > 0].copy()
chk["I_zarg_calc"] = chk.I_max / chk.I_dop * 100
chk["err"] = (chk.I_zarg - chk.I_zarg_calc).abs()
print("max |I_zarg - I_max/I_dop*100| (valid):", chk[chk.valid_exp == 1].err.max())
print("I_zarg > 100 % (перегрузка), доля среди valid:", (lines[lines.valid_exp == 1].I_zarg > 100).mean().round(4))

print("\nТоки по флагу:")
display(lines.groupby("valid_exp")[["I_max", "I_zarg"]].describe().T)
for c in ["I_max", "I_zarg"]:
    print(c, "valid=0: NaN", lines[lines.valid_exp == 0][c].isna().mean().round(3),
          "| нули", (lines[lines.valid_exp == 0][c] == 0).mean().round(3))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
d = lines[lines.valid_exp == 1]
sns.histplot(d.I_zarg.clip(0, d.I_zarg.quantile(.995)), bins=80, ax=ax[0]); ax[0].set_title("I_zarg, valid=1")
sns.histplot(d.I_max.clip(0, d.I_max.quantile(.995)), bins=80, ax=ax[1]); ax[1].set_title("I_max, valid=1")
d.groupby("line").I_zarg.mean().sort_values(ascending=False).head(25).plot.bar(ax=ax[2])
ax[2].set_title("Топ-25 линий по средней загрузке I_zarg")
plt.tight_layout(); plt.show()

## 6. Корреляции: входы → выходы

Строим «широкие» таблицы: 1 строка = 1 опыт, колонки = `<признак>_<номер узла/линии>`.

* **Входы:** `P_load`, `Q_load`, `P_gen` по узлам (+ параметры линий, если они меняются)
* **Выходы:** `V_pu`, `Delta` по узлам; `I_max`, `I_zarg` по линиям

In [ ]:
def wide(df, cols, key):
    w = df.set_index(["id", key])[cols].unstack(key)
    w.columns = [f"{a}_{b}" for a, b in w.columns]
    return w

node_in  = wide(nodes, ["P_load", "Q_load", "P_gen"], "node")
node_out = wide(nodes, ["V_pu", "Delta"], "node")
line_out = wide(lines, ["I_max", "I_zarg"], "line")

# параметры линий включаем как вход только если они реально меняются между опытами
varying = [c for c in ["R", "X", "B", "Kt", "I_dop"] if (lines.groupby("line")[c].nunique() > 1).any()]
line_in = wide(lines, varying, "line") if varying else pd.DataFrame(index=node_in.index)
print("Меняющиеся параметры линий:", varying or "нет (топология/параметры постоянны)")

ok = exp_valid[exp_valid == 1].index
drop_const = lambda df: df.loc[:, df.std() > 0]
X_nodes = drop_const(node_in.loc[ok])
Y_V     = drop_const(node_out.loc[ok, [c for c in node_out if c.startswith("V_pu")]])
Y_D     = drop_const(node_out.loc[ok, [c for c in node_out if c.startswith("Delta")]])
Y_I     = drop_const(line_out.loc[ok, [c for c in line_out if c.startswith("I_zarg")]])
print(X_nodes.shape, Y_V.shape, Y_D.shape, Y_I.shape)

In [ ]:
def cross_corr(X, Y, method="pearson"):
    return pd.concat([X, Y], axis=1).corr(method=method).loc[X.columns, Y.columns]

def top_pairs(c, k=30):
    s = c.stack().rename("corr").reset_index()
    s.columns = ["x", "y", "corr"]
    s["abs"] = s["corr"].abs()
    return s.sort_values("abs", ascending=False).head(k).reset_index(drop=True)

def heat(c, title, figsize=(13, 10)):
    plt.figure(figsize=figsize)
    sns.heatmap(c, cmap="coolwarm", center=0, vmin=-1, vmax=1, linewidths=.2)
    plt.title(title); plt.tight_layout(); plt.show()

C_V = cross_corr(X_nodes, Y_V)
C_D = cross_corr(X_nodes, Y_D)
C_I = cross_corr(X_nodes, Y_I)

print("Топ связей вход -> V_pu (valid=1):");  display(top_pairs(C_V, 25))
print("Топ связей вход -> Delta (valid=1):"); display(top_pairs(C_D, 25))
print("Топ связей вход -> I_zarg (valid=1):"); display(top_pairs(C_I, 25))

In [ ]:
for prefix, C, name in [("P_load", C_V, "V_pu"), ("P_gen", C_V, "V_pu"),
                        ("P_load", C_D, "Delta"), ("P_gen", C_D, "Delta")]:
    rows = [r for r in C.index if r.startswith(prefix)]
    heat(C.loc[rows], f"corr({prefix}_узла, {name}_узла), valid=1")

In [ ]:
# Какие входные группы вообще объясняют выходы: средняя |corr| по типу входа
summary = pd.DataFrame({
    "->V_pu":  [C_V.loc[[r for r in C_V.index if r.startswith(p)]].abs().mean().mean() for p in ["P_load", "Q_load", "P_gen"]],
    "->Delta": [C_D.loc[[r for r in C_D.index if r.startswith(p)]].abs().mean().mean() for p in ["P_load", "Q_load", "P_gen"]],
    "->I_zarg":[C_I.loc[[r for r in C_I.index if r.startswith(p)]].abs().mean().mean() for p in ["P_load", "Q_load", "P_gen"]],
}, index=["P_load", "Q_load", "P_gen"])
display(summary)

### 6.2. Линии ↔ узлы: что на что влияет

In [ ]:
# (а) Если параметры линий меняются - их связь с V, Delta, I
if varying:
    Xl = drop_const(line_in.loc[ok])
    C_lV = cross_corr(Xl, Y_V); C_lD = cross_corr(Xl, Y_D); C_lI = cross_corr(Xl, Y_I)
    print("параметры линий -> V_pu");  display(top_pairs(C_lV, 20))
    print("параметры линий -> Delta"); display(top_pairs(C_lD, 20))
    print("параметры линий -> I_zarg"); display(top_pairs(C_lI, 20))
else:
    print("Параметры линий константны -> влияют только через структуру сети (топологию), "
          "а в данных различаются лишь нагрузки/генерация.")

In [ ]:
# (б) Физическая проверка: ток линии vs напряжения/углы на концах.
# Эталонный узел 999 НЕ отбрасываем: для него нет V/Delta в таблице узлов, поэтому
# берём доступный конец линии, а для пары (разность по концам) - NaN.
li = line_info.reset_index()
vp = node_out.loc[ok]
res = []
for r in li.itertuples(index=False):
    cI = f"I_max_{r.line}"
    if cI not in line_out: continue
    I = line_out.loc[ok, cI]
    Vs, Ve = vp.get(f"V_pu_{r.n_s}"), vp.get(f"V_pu_{r.n_e}")
    Ds, De = vp.get(f"Delta_{r.n_s}"), vp.get(f"Delta_{r.n_e}")
    both = Vs is not None and Ve is not None
    res.append({"line": r.line, "n_s": r.n_s, "n_e": r.n_e, "type": r.type, "X": r.X,
                "has_ref999": REF in (r.n_s, r.n_e),
                "corr(I,|dDelta|)": I.corr((Ds - De).abs()) if both else np.nan,
                "corr(I,|dV|)":     I.corr((Vs - Ve).abs()) if both else np.nan,
                "corr(I,V_ns)": I.corr(Vs) if Vs is not None else np.nan,
                "corr(I,V_ne)": I.corr(Ve) if Ve is not None else np.nan,
                "corr(I,Delta_ns)": I.corr(Ds) if Ds is not None else np.nan,
                "corr(I,Delta_ne)": I.corr(De) if De is not None else np.nan})
phys = pd.DataFrame(res).set_index("line")
display(phys.sort_values("corr(I,|dDelta|)", ascending=False))

In [ ]:
# (в) Связь I_zarg линий с мощностью узлов на его концах
res = []
for r in li.itertuples(index=False):
    c = f"I_zarg_{r.line}"
    if c not in line_out: continue
    I = line_out.loc[ok, c]
    row = {"line": r.line, "n_s": r.n_s, "n_e": r.n_e}
    for p in ["P_load", "P_gen"]:
        for side in ["n_s", "n_e"]:
            col = f"{p}_{getattr(r, side)}"
            row[f"corr(I_zarg,{p}@{side})"] = I.corr(node_in.loc[ok, col]) if col in node_in and node_in.loc[ok, col].std() > 0 else np.nan
    res.append(row)
display(pd.DataFrame(res).set_index("line").sort_values("corr(I_zarg,P_load@n_s)", key=abs, ascending=False).head(30))

### 6.3. Эталонный узел 999

Линии, подключённые к 999, берут на себя небаланс режима. Смотрим их параметры, токи и связь с входами и с системным балансом.

In [ ]:
ref_lines = line_info[(line_info.n_s == REF) | (line_info.n_e == REF)]
print("Линии, подключённые к 999:", len(ref_lines))
display(ref_lines)

rl = ref_lines.index.tolist()
print("\nТоки этих линий по флагу:")
display(lines[lines.line.isin(rl)].groupby(["line", "valid_exp"])[["I_max", "I_zarg"]].agg(["mean", "std", "max"]))

# какие узлы-соседи 999
neigh = sorted(set(ref_lines.n_s) | set(ref_lines.n_e) - {REF})
neigh = sorted((set(ref_lines.n_s) | set(ref_lines.n_e)) - {REF})
print("Соседи 999:", neigh)

In [ ]:
# (а) связь токов линий к 999 с узловыми входами (valid=1)
Yr = drop_const(line_out.loc[ok, [f"I_zarg_{l}" for l in rl if f"I_zarg_{l}" in line_out]])
C_ref = cross_corr(X_nodes, Yr)
print("Топ связей: вход узла -> I_zarg линии к 999"); display(top_pairs(C_ref, 30))

# (б) токи в сторону 999 vs системный небаланс (генерация - нагрузка)
agg = pd.DataFrame({
    "sum_P_load": node_in.loc[ok].filter(like="P_load_").sum(axis=1),
    "sum_Q_load": node_in.loc[ok].filter(like="Q_load_").sum(axis=1),
    "sum_P_gen":  node_in.loc[ok].filter(like="P_gen_").sum(axis=1),
})
agg["gen_minus_load"] = agg.sum_P_gen - agg.sum_P_load
for l in rl:
    if f"I_max_{l}" in line_out: agg[f"I_max_line{l}"] = line_out.loc[ok, f"I_max_{l}"]
agg["sum_I_max_ref"] = agg.filter(like="I_max_line").sum(axis=1)
c = agg.corr()
print("Корреляция суммарного тока линий к 999 с балансом системы:")
display(c.loc[["sum_I_max_ref"] + [x for x in c.index if x.startswith("I_max_line")],
              ["sum_P_load", "sum_Q_load", "sum_P_gen", "gen_minus_load"]])

# (в) связь с V/Delta соседних узлов
for n in neigh:
    for nm in (f"V_pu_{n}", f"Delta_{n}"):
        if nm in node_out:
            print(nm, "vs I_max линий к 999:",
                  {l: round(line_out.loc[ok, f"I_max_{l}"].corr(node_out.loc[ok, nm]), 3) for l in rl if f"I_max_{l}" in line_out})

In [ ]:
# (г) Корреляция выходов друг с другом: V_pu / Delta / I_zarg (valid=1)
big = pd.concat([Y_V, Y_D, Y_I], axis=1)
print("Топ пар выход-выход:")
c_out = big.corr()
tri = c_out.where(np.triu(np.ones(c_out.shape), 1).astype(bool)).stack().rename("corr").reset_index()
tri.columns = ["a", "b", "corr"]
display(tri.reindex(tri["corr"].abs().sort_values(ascending=False).index).head(25))

## 7. Что влияет на сходимость (`regime_valid`)

Сравниваем входы валидных и невалидных опытов + быстрая модель (RandomForest) для важности признаков.

In [ ]:
feat = node_in.copy()
if varying: feat = feat.join(line_in)
feat["sum_P_load"] = node_in.filter(like="P_load_").sum(axis=1)
feat["sum_Q_load"] = node_in.filter(like="Q_load_").sum(axis=1)
feat["sum_P_gen"]  = node_in.filter(like="P_gen_").sum(axis=1)
feat["gen_minus_load"] = feat.sum_P_gen - feat.sum_P_load
feat["max_P_load"] = node_in.filter(like="P_load_").max(axis=1)
feat["n_gen_on"]   = (node_in.filter(like="P_gen_") != 0).sum(axis=1)
y = exp_valid.loc[feat.index]

# стандартизованная разница средних (valid=1 vs 0)
f = drop_const(feat)
smd = ((f[y == 1].mean() - f[y == 0].mean()) / f.std()).rename("std_mean_diff")
cor = f.corrwith(y).rename("corr_with_valid")
imp = pd.concat([smd, cor], axis=1)
print("Признаки, больше всего различающиеся между valid=1 и valid=0:")
display(imp.reindex(imp.std_mean_diff.abs().sort_values(ascending=False).index).head(25))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

Xtr, Xte, ytr, yte = train_test_split(f, y.loc[f.index], test_size=.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=200, max_depth=14, n_jobs=-1, random_state=42, class_weight="balanced")
rf.fit(Xtr, ytr)
pred = rf.predict(Xte)
print("F1-macro:", round(f1_score(yte, pred, average="macro"), 4))
print(classification_report(yte, pred))

fi = pd.Series(rf.feature_importances_, index=f.columns).sort_values(ascending=False)
fi.head(25).iloc[::-1].plot.barh(figsize=(8, 8), title="Важность признаков для regime_valid")
plt.tight_layout(); plt.show()

In [ ]:
# Распределения главных агрегатов по флагу
top = fi.head(6).index.tolist()
fig, ax = plt.subplots(2, 3, figsize=(17, 8))
for a, c in zip(ax.ravel(), top):
    d = pd.DataFrame({c: f[c], "valid": y.loc[f.index]})
    lo, hi = d[c].quantile([.005, .995])
    sns.histplot(data=d[d[c].between(lo, hi)], x=c, hue="valid", bins=60, stat="density", common_norm=False, ax=a)
    a.set_title(c)
plt.tight_layout(); plt.show()

## 8. Итоги (заполнить по результатам запуска)

* **Флаг:** гипотеза про единый флаг на опыт — см. раздел 1 (число смешанных опытов).
* **Q_н = k·P_н:** смотри `std(tg)` по узлам в разделе 2: ≈0 → константный коэффициент на узел.
* **P_г в 29/33:** раздел 3 (доля ненулевых, диапазон, связь между узлами).
* **Что на что влияет:** топ-пары в разделе 6; физическая проверка по линиям (6.2б).
* **Эталонный узел 999:** раздел 6.3 (токи линий к 999, связь с небалансом генерация − нагрузка и с соседними узлами).
* **Сходимость:** разница входов и важность признаков (раздел 7) — основа для фичей классификатора.